> 在线讲义：[第 6 章 · 动作当词：把每个数切成 256 格（RT-2 / OpenVLA）](https://wikieden.github.io/vla-from-scratch/lessons/06-action-tokens.html) · 本书《VLA 从零到 π0》

# 第 6 章 · 动作当词：把每个数切成 256 格（RT-2 / OpenVLA）

配套讲义：仓库里 `vla-deep-dive/notes/lessons/06-action-tokens.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `bin_width`、`to_bin` | 1 毫米；0 → 128、+0.02 → 148、−0.05 → 78、−0.1 → 28 |
| 第 1 步 | `bin_width(−0.128, 1.0)`、`quantile_range` | 4.4 毫米、58 格；分位数范围不被异常值撑大 |
| 第 2 步 | `decode_time(50, 7, 42)` | 8.3 秒 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

**函数 `bin_width`**：一格多宽。

In [ ]:
def bin_width(lo, hi, n_bins=256):
    """一格多宽。"""
    return (hi - lo) / n_bins

**函数 `to_bin`**：连续值 → 格子编号（0 到 n_bins − 1）。

In [ ]:
def to_bin(x, lo, hi, n_bins=256):
    """连续值 → 格子编号（0 到 n_bins − 1）。"""
    return int(np.clip(np.floor(round((x - lo) / bin_width(lo, hi, n_bins), 9)), 0, n_bins - 1))

**函数 `quantile_range`**：OpenVLA 的做法：用 1%–99% 分位数定范围，异常值撑不大格子。

In [ ]:
def quantile_range(values, q_lo=0.01, q_hi=0.99):
    """OpenVLA 的做法：用 1%–99% 分位数定范围，异常值撑不大格子。"""
    return float(np.quantile(values, q_lo)), float(np.quantile(values, q_hi))

**函数 `decode_time`**：逐词念完一个 chunk 要多少秒。

In [ ]:
def decode_time(chunk, dims, tokens_per_s):
    """逐词念完一个 chunk 要多少秒。"""
    return chunk * dims / tokens_per_s

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 连续的数 → 格子编号")
lo, hi = -0.128, 0.128
print(f"  范围 [−0.128, +0.128] 切 256 格：每格 {bin_width(lo, hi) * 1000:.1f} 毫米")
for x in (0.0, 0.02, -0.05, -0.1):
    print(f"  Δx = {x:+.2f} → 第 {to_bin(x, lo, hi)} 格")
w = bin_width(-0.128, 1.0)
print(f"  混进一个 +1.0 的异常值：每格 {w * 1000:.1f} 毫米，正常动作只用得上 {0.256 / w:.0f} 格")
rng = np.random.default_rng(0)
data = np.concatenate([rng.uniform(-0.128, 0.128, 999), [1.0]])
q = quantile_range(data)
print(f"  改用 1%–99% 分位数：范围 [{q[0]:+.3f}, {q[1]:+.3f}]，异常值不再撑大格子")

In [ ]:
print("\n第二步 逐词念：50 步 × 7 维")
print(f"  {50 * 7} 个词，每秒 42 个：{decode_time(50, 7, 42):.1f} 秒，而这段只管 1 秒")

## 3. 代码里才看得到的

1. `to_bin` 先 round 再 floor：0.02 / 0.001 在浮点里是 147.99999…，不 round 会落到 147。
2. OpenVLA 真实代码用 `np.digitize` 和 256 个刻度，返回 1–256，和这里 0–255 的约定差 1（源码注释里专门说明了这一点）。

## 4. 自己改一改

- 把格数改成 64，看每格多宽、精度掉多少。
- 给 `quantile_range` 换成 5%–95%，看范围怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 尺子切 256 格，小数变编号，编号当词说出来；
> 一个字一个字地念，念得太慢。